# Transfer Learning on Cats-Dogs Classification with HuggingFace Vision Transformer (ViT)

## Finetuning

### CIML Summer Institute
### UC San Diego

This notebook finetunes the ViT model from the Feature Extraction notebook to the new cats-vs-dogs task using HuggingFace Transformers.
Unlike feature extraction, finetuning updates more than just the classifier head, so the model can shift its internal representations toward the new dataset instead of relying only on fixed features.

We still use `AutoModelForImageClassification` because it gives us the standard Hugging Face image-classification interface, but here the important idea is that we start from our feature classification checkpoint and then continue training it on a smaller, task-specific dataset.

Base model:
https://huggingface.co/google/vit-base-patch16-224-in21k

`AutoModelForImageClassification` docs:
https://huggingface.co/docs/transformers/en/model_doc/auto#transformers.AutoModelForImageClassification

We read in the weights saved from our Feature Extraction notebook, but now we unfreeze more parameters and let the model further adapt to our cats-vs-dogs classification task.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# HuggingFace imports
from datasets import load_dataset, disable_caching
from transformers import DefaultDataCollator
from transformers import ViTImageProcessor, AutoModelForImageClassification
from transformers import TrainingArguments, Trainer
from transformers import pipeline
from transformers import EarlyStoppingCallback

# Image processing
from torchvision import transforms
from PIL import Image

# Evaluation
import evaluate

## Define Parameters

In [ ]:
# Image dimensions
IMAGE_DIM = 224
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

# Training parameters
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10
N_GPUS = torch.cuda.device_count()
N_CPUS = int(os.environ.get('SLURM_CPUS_ON_NODE', os.cpu_count()))

# Set seeds for reproducibility
seed = 42

# Data location
DATA_DIR = os.environ.get("CIML26_DATA_DIR") + "/catsVsDogs"

# Output directory for model checkpoints
OUTPUT_DIR = "vit_cats_dogs_model/finetune"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device {device} with {N_GPUS} GPUs and {N_CPUS} CPUs.")

In [ ]:
!jupyter --version
print (torch.__version__)
!python --version

# TODO:  Use linux shell command nvidia-smi to see GPU device.  
==> YOUR CODE HERE

## Load Data

The folder structure is turned into train, validation, and test splits with `datasets.load_dataset("imagefolder", ...)`.
That keeps the notebook focused on the modeling choices instead of custom file parsing.

Before we finetune, it is worth checking that the split sizes look right, because label and split mistakes usually come from the data layout rather than the model.

In [ ]:
# Load the dataset
data = load_dataset("imagefolder", data_dir=DATA_DIR)

# Print dataset information
print(f"Train dataset size: {len(data['train'])}")
print(f"Validation dataset size: {len(data['validation'])}")

# TODO: Print information about the length of the test dataset
# HINT replicate the print above but for data['test']
==> YOUR CODE HERE

## Define Image Transformations

Finetuning usually benefits from stronger preprocessing than feature extraction because the classifier is no longer the only part learning task-specific patterns.
Here we pair two ideas: a deterministic `ViTImageProcessor` for resize and normalization, and a separate augmentation pipeline for training images only.

The data augmentation randomly alters are training images, helping the model see slightly different versions of the same image during finetuning, which improves robustness and makes better use of a smaller dataset.


In [ ]:
# Load the image processor
image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")

# Keep augmentation geometric and let image_processor handle resize/normalize.
train_augmentation = transforms.Compose(
    [
        transforms.RandomAffine(degrees=0, shear=0.2),
        transforms.RandomResizedCrop(size=IMAGE_DIM, scale=(0.8, 1.2)),

        # TODO: add a transform here that randomly flips images horizontally 
        # HINT: use torchvision.transforms.RandomHorizontalFlip()
        ==> YOUR CODE HERE
    ]
)

# Batched preprocessing function for training data
def train_transform(example_batch):
    images = [train_augmentation(image.convert("RGB")) for image in example_batch["image"]]
    inputs = image_processor(images=images, return_tensors="pt")
    inputs["labels"] = example_batch["label"]
    return inputs

# Batched preprocessing function for validation and test data
def eval_transform(example_batch):
    images = [image.convert("RGB") for image in example_batch["image"]]
    inputs = image_processor(images=images, return_tensors="pt")
    inputs["labels"] = example_batch["label"]
    return inputs

In [ ]:
from transformers import set_seed
set_seed(seed)

processed_data = {}
processed_data["train"] = data["train"].with_transform(train_transform)
processed_data["validation"] = data["validation"].with_transform(eval_transform)
processed_data["test"] = data["test"].with_transform(eval_transform)

In [ ]:
print("RAW train columns:", data["train"].column_names)

# TODO: Print the feature labels in the dataset (Should have 'cats' and 'dogs')
# HINT: Replicate the print statement above, but instead with data['train'].features

batch = data["train"][:1]
print("Example keys in raw batch:", batch.keys())

# Manually run transform once
print("Keys returned by train_transform:", train_transform(batch).keys())

## Set Up Model

In finetuning, we do not want to retrain the whole network from scratch, but we do want more of the model to adapt than in feature extraction.
That usually means starting from a trained checkpoint, freezing everything first, and then selectively unfreezing the classifier and the top encoder layers so the representation can move toward the new task.

This gives you a practical middle ground: the backbone still begins with useful visual features, but the model can adjust the higher-level layers to better separate cats and dogs.

**TODO**: In the cell below:
1. Get labels from the dataset and create `id2label` and `label2id` mappings
2. Load the pre-trained model from the feature extraction checkpoint
3. Freeze all parameters first
4. Unfreeze the classifier head and the last encoder layer
5. Print model architecture summary

In [ ]:
# TODO: Complete this cell

# Step 1: Get the labels from the dataset
labels = data['train'].features['label'].names
id2label = {i: label for i, label in enumerate(labels)}

# TODO: Create a reverse mapping from labels to id number
# Hint use the id2label mapping defined above to create the reverse mapping
label2id = ==> YOUR CODE HERE


# Step 2: Load the pre-trained model
model = AutoModelForImageClassification.from_pretrained(
    "vit_cats_dogs_model/feature_extraction_best_model",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# Step 3: Freeze everything first
# TODO: Set param.requires_grad to false to freeze the parameter
for param in model.parameters():
    ==> YOUR CODE HERE

# Step 4: Unfreeze classifier head and last encoder layer
# TODO: Reset param.requires_grad to true for the layers that we want to train
for name, param in model.named_parameters():
    if name.startswith("classifier") or "encoder.layer.11" in name:
        ==> YOUR CODE YERE

# Step 5: Print model architecture summary
print(model)
total_params = sum(p.numel() for p in model.parameters())

# TODO: Count number of trainable parameters in the model
# HINT: Code with be similar to counting total params but use a clause to only count if p.requires_grad is true
trainable_params = ==> YOUR CODE HERE

# TODO: Count number of frozen parameters
# HINT: The number of frozen parameters can be expressed as total_params - trainable params
frozen_params = ==> YOUR CODE HERE

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params:.2%})")
print(f"Frozen parameters: {frozen_params:,} ({frozen_params/total_params:.2%})")

print(f"Number of output neurons: {model.classifier.out_features}")


## Set Up Training

`Trainer` still handles the training loop, evaluation, and checkpointing, but finetuning usually needs a bit more care than linear probing.
Because more layers are trainable, the learning rate, precision setting, and early stopping behavior matter more, and the notebook is set up to surface those choices clearly.

The main question now is not just whether the head learns, but whether adapting the backbone improves validation performance without overfitting too quickly.

To initialize the `Trainer`, you can check the documentation: https://huggingface.co/docs/transformers/main_classes/trainer

In [ ]:
# Define metrics for evaluation
metric = evaluate.load("accuracy")

# In Huggingface, compute_metrics is passed as a callable argument to the Trainer. It runs during evaluation to turn model predictions and labels into metrics (e.g., accuracy, F1) that appear in the eval logs and results.
def compute_metrics(eval_pred):
    """Compute metrics for evaluation"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

# Set up training arguments. This object will be passed in to the "args" parameter of our Trainer object
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch", 
    logging_strategy = "epoch",
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    
    # Performance optimizations
    bf16=True,                            # use bfloat16 on modern GPUs
    dataloader_num_workers=N_CPUS,        # Multi-threaded data loading

    remove_unused_columns=False,
    
    push_to_hub=False,
    report_to="none",
)

callbacks = [EarlyStoppingCallback (early_stopping_patience=3, early_stopping_threshold=0.001)]

# Create data collator
data_collator = DefaultDataCollator()

In [ ]:
# TODO: Initialize the Trainer
# HINT: The Trainer takes in all of the necessary arguments for training. 
#       You should set the following arguments in the Trainer. 

trainer = Trainer(
    model= ==> YOUR CODE HERE
    args= ==> YOUR CODE HERE
    train_dataset=processed_data['train'],
    eval_dataset= ==> YOUR CODE HERE
    data_collator= ==> YOUR CODE HERE
    compute_metrics= ==> YOUR CODE HERE
    callbacks= ==> YOUR CODE HERE
)


## Train Model

This is the finetuning step where the model updates the unfrozen parameters on the cats-vs-dogs data.
Compared with feature extraction, you should expect training to be a little slower, but also more flexible because the backbone can move toward the task instead of staying fixed.

Use the validation metrics to judge whether the extra trainable capacity is actually helping.

In [ ]:
# Train the model
train_results = trainer.train()

# Print training metrics
print(f"Training results: {train_results}")

# Save the model
trainer.save_model("vit_cats_dogs_model/finetune_best_model")
trainer.save_state()

## Evaluate Model

At this point the goal is to measure the benefit of finetuning, not just to see whether the model memorized the training set.
The classification report gives per-class precision and recall so you can tell whether the adapted backbone is improving one class more than the other.

Let's evaluate the trained model on every split and compare the results across train, validation, and test.

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Use trainer.predict() to get predictions for each split
for split_key in processed_data.keys():
    # Get predictions and labels
    predictions = trainer.predict(test_dataset=processed_data[split_key])
    y_pred = np.argmax(predictions.predictions, axis=1)
    y_true = predictions.label_ids
    
    # Generate sklearn classification report
    print(f"{split_key.capitalize()}:")
    print(classification_report(y_true, y_pred, digits=4))
    print()

## Perform Inference

Once finetuning is done, inference should reuse the same Hugging Face preprocessing and label decoding path as training.
That makes it easy to confirm the checkpoint is behaving as expected on individual images and to inspect the model’s predictions visually.

Let's use the trained model to make predictions on new images.

In [ ]:
# Create an inference pipeline
classifier = pipeline(
    "image-classification", 
    model=model, 
    image_processor=image_processor
)

from PIL import Image
import matplotlib.pyplot as plt

def predict_image(image_path):
    """Make a prediction on a single image and display it with the inference result."""
    image = Image.open(image_path)
    result = classifier(image)
    # Show the image and prediction
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title(f"Prediction: {result[0]['label']} ({result[0]['score']:.2f})")
    plt.axis('off')
    plt.show()
    return result

In [ ]:
# TODO: Perform inference on this image using predict_image()
image_path = DATA_DIR + "/test/cats/cat.1070.jpg"
==> YOUR CODE HERE

In [ ]:
# TODO: Perform inference on this image using predict_image()
image_path = DATA_DIR + "/test/dogs/dog.1233.jpg"
==> YOUR CODE HERE

In [ ]:
# TODO: Performance inference on this test image: cat.1080.jpg
==> YOUR CODE HERE

In [ ]:
# TODO: Perform inference on this test image: dog.1132.jpg
==> YOUR CODE HERE